In [5]:
import numpy as np
import numpy.testing as npt
from scipy import stats
from scipy.stats import f
from scipy.stats import f_oneway
import statsmodels.api as sm
import pandas as pd
from patsy import dmatrices
from numpy.testing import assert_almost_equal, assert_allclose
import matplotlib.pyplot as plt
import seaborn as sns
import sys
from copy import deepcopy
sys.path.insert(1, r'C:\Users\TODO\Desktop\Abhi\AI\AI\Math\Hands-On\statemodelsStudy')
from olsRegressionAnalysis import dispAnalysisOfVariance, tableDispFormatt,getInvOfProductMat,\
                                  getRegressionEqn,\
                                  dispReghressionAnalysis,norm_scalling,getCorrelation,\
                                  get_variance_inflation_factors,goodnessOfFitTestOfParams

In [6]:
path = r'C:\Users\TODO\Desktop\Abhi\AI\AI\allDataSet\STAT501_Lesson05\STAT501_Lesson05\allentest.txt'
df = pd.read_csv(path)
# https://online.stat.psu.edu/stat501/lesson/6/6.3


### Extra SumOf Square Method:

-----------------------------------------------------------------------

- the error sum of squares has been reduced,
- the regression sum of squares has increased,
- the total sum of squares stays the same.

y = β0 + β1x1 + β2x2 +  β3x3 + β4x4 + β5x5 +  ε 

**1. One degree of freedome:**

- SS(X3|X1) or SS(β3|β0,β3) -> We are calculating effect due to X3 when
                                 X1 already in models. 

So F = (SSr(X3|X1)/r)/MSres

    r = DoF = 1 (One degree of freedome)

    MSres = Due to combine effect of X1 & X3
       
**2. Two degree of freedome:**  

    SS(X2X3|X1) or SS(β2β3|β0,β1)      
    r = 2    

**2. three degree of freedome:**   

    SS(X2X3X4|X1) or SS(β2β3β4|β0,β1)  

    r = 3             

------------------------------------------------------------------------

Example taken from here  https://online.stat.psu.edu/stat501/lesson/6/6.3

**Let your models:**

y = β0 + β1x1 + β2x2 +  β3x3 + β4x4 + β5x5 +  ε 

Let suppos i wnat to calculate effect of independent variable X3 if

X1 already given in models. Calculate SS(X3|X1).

**Calculate SS(X3|X1) or SS(β3|β0,β3)**

- Step 1: Caluclate SSres/MSres & SSr/MSr due to X1 for this
models

y = β0 + β1x1 -> this will give SS(β0,β1)

- Step 2: Caluclate SSres/MSres & SSr/MSr due to X1&X2 for this
models

y = β0 + β1x1 + β3x3 -> this wil give SS(β0,β1,β3)

**Calulate MSres due to residuals.**

Here you can see if Sum of square residuals/regressor are redusing or
incresing. Remember for good modells Sum of square should be Minimum.

- Step 3: 

SSr -> Sum of square due to regressor/indepenmdent variabls group

SSr(X3|X1) = SSr(β3|β0,β3) = SSr(β0,β1,β3) - SSr(β0,β1)

We reduse extra sum from models. That why called extra sum of square

- Step 4:

**Calculate F score**

F = (SSr(β3|β0,β3)/r)/MSres

where r = number of regressor, Here r = 1 due to β3

MSres = calcuated in step 2.

- Step 5:

Write Null hypthesis & conclude result

- Step 6:

Plot difference SSr


In [7]:
idx = ['NOBS','DofModels','DofErr','SSres','SSr','MSres','MSr','rSqr','rSqrAdj','fVal','fPval']

# Step 1: Calculate SSr due to Vocab
dfX = pd.DataFrame({})
y, X = dmatrices('ACL ~ Vocab', data=df, return_type='dataframe')
mod = sm.OLS(y, X)    # Describe model
res = mod.fit()       # Fit model
SSres_X1 = res.ssr
MSres_X1 = res.mse_model
SSr_X1   = res.ess
MSr_X1   = res.mse_resid
dfSer = pd.Series(np.array([res.nobs,
                            res.df_model,res.df_resid,
                            res.ssr,res.ess,
                            res.mse_model,res.mse_resid,
                            res.rsquared,res.rsquared_adj,
                            res.fvalue,res.f_pvalue]
                           ),
                  index = idx)
dfX['X1'] = dfSer


### Step 1: Conclusions:

SSr (regressor sumof square ) is incresed by 2.690602 or Error/Residuals
sum of square (SSres) decreased by 40.358963 wehn we add Only regressor
Vocab

### Step 2: Calculate SSr due to Vocab & SDMT

In [8]:
y, X = dmatrices('ACL ~ Vocab + SDMT', data=df, return_type='dataframe')
mod = sm.OLS(y, X)    # Describe model
res = mod.fit()       # Fit model

SSres_X1X3 = res.ssr
MSres_X1X3 = res.mse_model
SSr_X1X3   = res.ess
MSr_X1X3   = res.mse_resid

dfSer = pd.Series(np.array([res.nobs,
                            res.df_model,res.df_resid,
                            res.ssr,res.ess,
                            res.mse_model,res.mse_resid,
                            res.rsquared,res.rsquared_adj,
                            res.fvalue,res.f_pvalue]
                           ),
                  index = idx)
dfX['X1X3'] = dfSer
print('X1 == Vocab')
print('X3 == SDMT')
print(dfX)

tableDispFormatt('Sum of square')
print('SSresX3|X1: ',SSres_X1 - SSres_X1X3 )
print('SSrX3|X1: ', SSr_X1X3 - SSr_X1)

X1 == Vocab
X3 == SDMT
                  X1       X1X3
NOBS       69.000000  69.000000
DofModels   1.000000   2.000000
DofErr     67.000000  66.000000
SSres      40.358963  31.271729
SSr         2.690602  11.777836
MSres       2.690602   5.888918
MSr         0.602373   0.473814
rSqr        0.062500   0.273588
rSqrAdj     0.048508   0.251575
fVal        4.466675  12.428753
fPval       0.038290   0.000026
=============================== Sum of square ==============================================
SSresX3|X1:  9.087234022238711
SSrX3|X1:  9.087234022238711



### Conclusions:

9.0872 is the reduction in the error sum of squares — or the 
increase in the regression sum of squares — when you add x3 = SDMT 
to a model already containing x1 = Vocab. That is, 9.0872 is the 
***sequential sum of squares SSR(x3|x1)***

### Step 4: Calculate F score

In [9]:
r = 1 # due only one Group i.e. 'X3'
F = ((SSr_X1X3 - SSr_X1)/r)/MSr_X1X3
print('Fstae: ',F)

# Step 5: Calculate hypothesis value
tableDispFormatt('null hypthesis')

Fstae:  19.178902680697888
=============================== null hypthesis =============================================


In [10]:
######################################################
######################################################
######################################################
#~~~~~~~~~~~~~~~ Order matters ~~~~~~~~~~~~~~~~~~~~~~#
######################################################
######################################################
######################################################
######################################################

'''
We are taking same previous exampls. Just order change
'''
# First calculate regressing y = ACL on X3 = SDMT 

y, X = dmatrices('ACL ~ SDMT', data=df, return_type='dataframe')
mod = sm.OLS(y, X)    # Describe model
res = mod.fit()

SSres_X3 = res.ssr
MSres_X3 = res.mse_model
SSr_X3   = res.ess
MSr_X3   = res.mse_resid

### First calculate regressing y = ACL on X3 = SDMT and X1 = Vocab

In [11]:

y, X = dmatrices('ACL ~ SDMT + Vocab', data=df, return_type='dataframe')
mod = sm.OLS(y, X)    # Describe model
res = mod.fit()

SSres_X3X1  = res.ssr
MSres_X3X1  = res.mse_model
SSr_X3X1    = res.ess
MSr_X3X1    = res.mse_resid


SSR(x1|x3) = 0.0979. That is, the error sum of squares is reduced — 
or the regression sum of squares is increased — by (only!) 0.0979 when you add 
x1 = Vocab to a model already containing x3 = SDMT.

***Note***:
 SSr or SSred not much affected when we add Vocab regressor with SDMT.
 So again we coclude Vocab not play much roal in compaire SDMT



### Qus 1:

 y = ACL on 

 x3 = SDMT and 

 x1 = Vocab and 

 x2 = Abstract
 
 Find SSr(x2|x1,x3)


In [12]:
######################################################
######################################################
######################################################
#Two- (or three- or more-) degree of freedom sequential sums of squares#
######################################################
######################################################
######################################################
######################################################


y, X = dmatrices('ACL ~ Vocab + SDMT', data=df, return_type='dataframe')
mod = sm.OLS(y, X)    # Describe model
res_1 = mod.fit()
y, X = dmatrices('ACL ~ Vocab + SDMT + Abstract', data=df, return_type='dataframe')
mod = sm.OLS(y, X)    # Describe model
res_2 = mod.fit()
print('SSr x2|x1x3 : ',res_2.ess - res_1.ess)
print('SSres x2|x1x3 : ',res_1.ssr - res_2.ssr)

SSr x2|x1x3 :  0.5230296885386636
SSres x2|x1x3 :  0.5230296885386636



### Qus 2:

 y = ACL on 

 x3 = SDMT and 

 x1 = Vocab and 

 x2 = Abstract
 
 Find SSr(x1,x2|x3) 


In [13]:
y, X = dmatrices('ACL ~ SDMT', data=df, return_type='dataframe')
mod = sm.OLS(y, X)    # Describe model
res_1 = mod.fit()
y, X = dmatrices('ACL ~ Vocab + SDMT + Abstract', data=df, return_type='dataframe')
mod = sm.OLS(y, X)    # Describe model
res_2 = mod.fit()

print('SSr x1,x2|x3 : ',res_2.ess - res_1.ess)
print('SSres x1,x2|x3 : ',res_1.ssr - res_2.ssr)

# Step 4: Calculate F score
r = 2 # due only one Group i.e. 'X1,x2'
F = ((res_2.ess - res_1.ess)/r)/res_2.mse_resid
print('Fstae: ',F)

SSr x1,x2|x3 :  0.6209748601406275
SSres x1,x2|x3 :  0.6209748601406275
Fstae:  0.6563426572178545
